In [ ]:
# This file will handle new incoming data for orders
# Saving the original data to the bronze table then procesing it through staging stages within a bronze and silve staging table
# New data is then merged with the existing data that resides in the gold.fact_orders

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3//:sports-consolidated-dp/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"


In [ ]:
# Read from the 'landing' folder from S3 bucket and forward it into the staging
# Landing folder is new incoming data to be appended to the Bronze
df = spark.read.options(header=True, inferSchema=True).csv(f'{landing_path}/*.csv').withColumn('read_timestamp', F.current_timestamp()), '_metadata.file_name', '_metadata.file_size'

print('Total Rows: ', df.count())
df.show(5)
# Total Rows:  8834
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FDEC818102602|Tuesday, December...|     789102|  25891502|    196.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891503|     NULL|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891602|    147.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|     789102|  25891101|    337.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# |FDEC818102602|Tuesday, December...|    INVALID|  25891202|    202.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 5 rows

In [ ]:
# Append new incoming data along with existing data to bronze table
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('append') \
    .saveAsTable(bronze_table)
    

In [ ]:
# Staging table to process the new ariving incremental data
df.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.staging_{data_source}')

In [ ]:
# Moving files from source to processed directory
files = dbutils.fs.ls(landing_path)
for file in files:
    dbutils.fs.mv(
        file.path,
        f'{processed_path}/{file.name}',
        True
    )

In [ ]:
# ---> SILVER <---
df_orders = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.staging_{data_source};')
df_orders.show(2)

# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# |FDEC85903601|Thursday, Decembe...|     789903|  77777777|    342.0|2025-11-24 14:09:...|orders_2025_12_04...|    22385|
# |FDEC85903601|Thursday, Decembe...|     789903|  25891302|     53.0|2025-11-24 14:09:...|orders_2025_12_04...|    22385|
# +------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
# only showing top 2 rows

In [ ]:
# Keep rows where order_qty is present
df_orders = df.orders.filter(F.col('order_qty').isNotNull())

# Clean customer_id: keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
     .otherwise('999999')
     .cast('string')
)

# Remove weekday name from the date text
# 'Tuesday, July 01, 2025' -> 'July 01, 2025
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*', '')
)

# Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
    )
)

# Drop duplicates
df_orders = df_orders.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])

# Conver product_id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [ ]:
# Check the max and min date
df_orders.agg(
    F.min('order_placement_date').alias('min_date'),
    F.min('order_placement_date').alias('max_date')
)

# +----------+----------+
# |  min_date|  max_date|
# +----------+----------+
# |2025-12-04|2025-12-30|
# +----------+----------+

In [ ]:
# JOIN WITH PRODUCTS
df_products = spark.table('fmcg.silver.products')
df_joined = df_orders.join(df_products, on='product_id', how='inner').select(df_orders['*'], df_products['product_code'])

df_joined.show(5)

# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
# |     order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|        product_code|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
# | FDEC87522503|          2025-12-04|     789522|  25891403|    342.0|2025-11-24 14:09:...|orders_2025_12_04...|    22385|77b6f538a9d0e0cf8...|
# | FDEC89522601|          2025-12-08|     789522|  25891403|    476.0|2025-11-24 14:09:...|orders_2025_12_08...|    21711|77b6f538a9d0e0cf8...|
# |FDEC817203502|          2025-12-15|     789203|  25891203|    300.0|2025-11-24 14:09:...|orders_2025_12_15...|    20287|889c67757ece9c973...|
# |FDEC818101501|          2025-12-16|     789101|  25891501|    221.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|ee1f7df9cf660ef02...|
# |FDEC819101203|          2025-12-16|     789101|  25891203|    393.0|2025-11-24 14:09:...|orders_2025_12_16...|    23060|889c67757ece9c973...|
# +-------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+--------------------+
# only showing top 5 rows

In [ ]:
# UPSERT NEW INCREMENTAL DATA INTO THE SILVER TABLE

if not(spark.catalog.tableExists(silver_table)):
    df_joined.write.format('delta').option(
        'delta.enableChangeDatFeed', 'true'
    ).option('mergeSchema', 'true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_table = DeltaTable(spark, silver_table)
    silver_table.alias('parent').join(df_joined.alias('child'), 'parent.order_placement_date = child.order_placement_date AND parent.order_id = child.order_id AND parent.product_code = child.product_code AND parent.customer_id = child.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute

In [ ]:
# NEW SILVER_STAGING TABLE TO PROCESS THE NEWLY INCREMENTAL DATA

df_joined.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{silver_schema}.staging_{data_source}')

In [ ]:
# ---> Gold <---
df_gold = spark.sql(f'SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, order_qty as sold_quantity FROM {catalog}.{silver_schema}.staging_{data_source};')
df_gold.show(2)

# +------------+----------+-------------+--------------------+----------+-------------+
# |    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
# +------------+----------+-------------+--------------------+----------+-------------+
# |FDEC87522503|2025-12-04|       789522|77b6f538a9d0e0cf8...|  25891403|        342.0|
# |FDEC89522601|2025-12-08|       789522|77b6f538a9d0e0cf8...|  25891403|        476.0|
# +------------+----------+-------------+--------------------+----------+-------------+
# only showing top 2 rows)

In [ ]:
df_gold.count()
# 6947

# UPSERT TO GOLD COMPANY TABLE FOR FACT_ORDERS
if not(spark.catalog.tableExists(gold_table)):
    print('Creating New Table')
    df_gold.write.format('delta').option(
        'delta.enableChangeDataFeed', 'true'
    ).option('mergeSchema', 'true').mode('overwrite').saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias('parent').merge(df_gold.alias('child'), 'parent.date = child.date AND parent.order_id = child.order_id AND parent.product_code = child.product_code AND parent.customer_code = child.customer_code').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
# MERGING WITH PARENT COMPANY

# Required: data for monthly bases but child data is viewed as daily

# Incremental load

# df_child: The incremendtal daily rows

df_child = spark.sql(f'SELECT order_replacement_date as date FROM {catalog}.{silver_schema}.staging_{data_souce};')

incremental_month_df = df_child.select(
    F.trunc('date', 'MM').alias('start_month')
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView('incremental_months')

# +-----------+
# |start_month|
# +-----------+
# | 2025-12-01|
# +-----------+

In [ ]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {catalog}.gold_schema.sb_fact_orders sb
    INNER JOIN incremental_momths m
        ON trunc(sb.date, 'MM') = m.start_month
""")

print('Total Rows: ', monthly_table.count())
monthly_table.show(10)

# Total Rows:  7736
# +----------+--------------------+-------------+-------------+
# |      date|        product_code|customer_code|sold_quantity|
# +----------+--------------------+-------------+-------------+
# |2025-12-04|77b6f538a9d0e0cf8...|       789522|        342.0|
# |2025-12-08|77b6f538a9d0e0cf8...|       789522|        476.0|
# |2025-12-15|889c67757ece9c973...|       789203|        300.0|
# |2025-12-16|ee1f7df9cf660ef02...|       789101|        221.0|
# |2025-12-16|889c67757ece9c973...|       789101|        393.0|
# |2025-12-21|ee1f7df9cf660ef02...|       789622|        190.0|
# |2025-12-05|e91ba9d665f90254d...|       789303|        462.0|
# |2025-12-14|889c67757ece9c973...|       789702|        322.0|
# |2025-12-23|e91ba9d665f90254d...|       789201|        343.0|
# |2025-12-26|3cab59f0592428527...|       789702|         37.0|
# +----------+--------------------+-------------+-------------+
# only showing top 10 rows

In [ ]:
monthly_table.select('date').distinct().orderBy('date').show()
# +----------+
# |      date|
# +----------+
# |2025-12-01|
# |2025-12-02|
# |2025-12-03|
# |2025-12-04|
# |2025-12-05|
# |2025-12-06|
# |2025-12-07|
# |2025-12-08|
# |2025-12-09|
# |2025-12-10|
# |2025-12-11|
# |2025-12-12|
# |2025-12-13|
# |2025-12-14|
# |2025-12-15|
# |2025-12-16|
# |2025-12-17|
# |2025-12-18|
# |2025-12-19|
# |2025-12-20|
# +----------+
# only showing top 20 rows

In [ ]:
# Group by and sum up the 'sold_quantity' of products purchased by customers on a monthly basis

df_monthly_recalc = (
    monthly_table
    .withColumn('month_start', F.trunc('date', 'MM'))
    .groupBy('month_start', 'product_code', 'customer_code')
    .agg(F.sum('sold_quantity').alias('sold_quantity'))
    .withColumnRenamed('month_start', 'date')
)

df_monthly_recalc.show(10, truncate=False)

df_monthly_recalc.count()

# +----------+----------------------------------------------------------------+-------------+-------------+
# |date      |product_code                                                    |customer_code|sold_quantity|
# +----------+----------------------------------------------------------------+-------------+-------------+
# |2025-12-01|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|789402       |1096.0       |
# |2025-12-01|778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6|789503       |1839.0       |
# |2025-12-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789703       |1759.0       |
# |2025-12-01|3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345|789103       |686.0        |
# |2025-12-01|889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2|789402       |3340.0       |
# |2025-12-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789321       |3765.0       |
# |2025-12-01|2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896|789101       |2857.0       |
# |2025-12-01|716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742|789603       |896.0        |
# |2025-12-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789902       |4737.0       |
# |2025-12-01|102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268|789221       |5903.0       |
# +----------+----------------------------------------------------------------+-------------+-------------+
# only showing top 10 rows

# 612


In [ ]:
gold_parent_delta = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.fact_orders')
gold_parent_delta.alias('parent').merge(df_monthly_recalc.alias('child'), 'parent.date = child.date AND parent.product_code = child.product_code AND parent.customer_code = child.customer_code').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [ ]:
# CLEAN UP THE STAGING TABLES
%sql
DROP TABLE fmcg.bronze.staging_orders;

%sql
DROP TABLE fmcg.silver.staging_orders;